# Matrix Factorization using SVD

Build recommendation system using Singular Value Decomposition (SVD).

In [ ]:
import sys
sys.path.append('../src')
sys.path.append('../evaluation')

import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from metrics import rmse, mae, precision_at_k, recall_at_k, ndcg_at_k
import matplotlib.pyplot as plt

## 1. Load Data

In [ ]:
# Load data
ratings = pd.read_csv('../data/processed/ratings.csv')
user_item_matrix = pd.read_csv('../data/processed/user_item_matrix.csv', index_col=0)

print(f"Data shape: {user_item_matrix.shape}")

## 2. Split Train-Test

In [ ]:
# Split data
train_ratings, test_ratings = train_test_split(ratings, test_size=0.2, random_state=42)

# Create matrices
train_matrix = train_ratings.pivot_table(index='user_id', columns='item_id', values='rating').fillna(0)
test_matrix = test_ratings.pivot_table(index='user_id', columns='item_id', values='rating').fillna(0)

print(f"Train shape: {train_matrix.shape}")
print(f"Test shape: {test_matrix.shape}")

## 3. SVD Matrix Factorization

In [ ]:
class SVDRecommender:
    """Matrix factorization using SVD"""
    
    def __init__(self, n_factors=50):
        self.n_factors = n_factors
        self.svd = TruncatedSVD(n_components=n_factors, random_state=42)
        self.user_factors = None
        self.item_factors = None
        self.user_ids = None
        self.item_ids = None
    
    def fit(self, matrix):
        """Fit SVD model"""
        self.user_ids = matrix.index.tolist()
        self.item_ids = matrix.columns.tolist()
        
        # Apply SVD
        self.user_factors = self.svd.fit_transform(matrix)
        self.item_factors = self.svd.components_.T
        
        return self
    
    def predict(self, user_id, item_id):
        """Predict rating for user-item pair"""
        if user_id not in self.user_ids or item_id not in self.item_ids:
            return 3.0  # Default rating
        
        user_idx = self.user_ids.index(user_id)
        item_idx = self.item_ids.index(item_id)
        
        pred = np.dot(self.user_factors[user_idx], self.item_factors[item_idx])
        
        # Clip to valid rating range
        return np.clip(pred, 0.5, 5.0)
    
    def recommend_for_user(self, user_id, n_items=10):
        """Recommend top-N items for user"""
        if user_id not in self.user_ids:
            return []
        
        user_idx = self.user_ids.index(user_id)
        
        # Get predicted ratings for all items
        predictions = np.dot(self.user_factors[user_idx], self.item_factors.T)
        
        # Get top-N items
        top_indices = np.argsort(predictions)[::-1][:n_items]
        
        return [self.item_ids[idx] for idx in top_indices]

# Train SVD model
svd_model = SVDRecommender(n_factors=50)
svd_model.fit(train_matrix)

print("SVD model trained!")
print(f"User factors shape: {svd_model.user_factors.shape}")
print(f"Item factors shape: {svd_model.item_factors.shape}")

## 4. Evaluate SVD Model

In [ ]:
# Evaluate on test set
predictions = []
actual = []

for _, row in test_ratings.iterrows():
    user_id = row['user_id']
    item_id = row['item_id']
    actual_rating = row['rating']
    
    if user_id in svd_model.user_ids and item_id in svd_model.item_ids:
        pred_rating = svd_model.predict(user_id, item_id)
        predictions.append(pred_rating)
        actual.append(actual_rating)

if predictions:
    rmse_score = rmse(actual, predictions)
    mae_score = mae(actual, predictions)
    
    print(f"SVD Model Results:")
    print(f"RMSE: {rmse_score:.4f}")
    print(f"MAE: {mae_score:.4f}")
    print(f"Test samples: {len(predictions)}")

## 5. Explore Latent Factors

In [ ]:
# Visualize variance explained by factors
explained_variance = svd_model.svd.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, len(explained_variance) + 1), explained_variance)
plt.xlabel('Factor')
plt.ylabel('Explained Variance Ratio')
plt.title('Explained Variance by Each Factor')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance)
plt.axhline(y=0.9, color='r', linestyle='--', label='90% Variance')
plt.xlabel('Number of Factors')
plt.ylabel('Cumulative Explained Variance')
plt.title('Cumulative Explained Variance')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"Variance explained by 50 factors: {cumulative_variance[-1]:.4f}")
print(f"Factors needed for 90% variance: {np.argmax(cumulative_variance >= 0.9) + 1}")

## 6. Get Recommendations

In [ ]:
# Get recommendations for sample users
for user_id in [1, 5, 10]:
    recommendations = svd_model.recommend_for_user(user_id, n_items=10)
    print(f"Top 10 recommendations for User {user_id}: {recommendations}")

## 7. Hyperparameter Tuning

In [ ]:
# Test different number of factors
factors_list = [10, 20, 30, 40, 50, 60, 70]
rmse_scores = []

for n_factors in factors_list:
    model = SVDRecommender(n_factors=n_factors)
    model.fit(train_matrix)
    
    predictions = []
    actual = []
    
    for _, row in test_ratings.head(500).iterrows():
        user_id = row['user_id']
        item_id = row['item_id']
        actual_rating = row['rating']
        
        if user_id in model.user_ids and item_id in model.item_ids:
            pred_rating = model.predict(user_id, item_id)
            predictions.append(pred_rating)
            actual.append(actual_rating)
    
    rmse_score = rmse(actual, predictions)
    rmse_scores.append(rmse_score)
    print(f"Factors: {n_factors}, RMSE: {rmse_score:.4f}")

# Plot results
plt.figure(figsize=(10, 5))
plt.plot(factors_list, rmse_scores, marker='o', linewidth=2)
plt.xlabel('Number of Factors')
plt.ylabel('RMSE')
plt.title('SVD Performance vs Number of Factors')
plt.grid(True)
plt.tight_layout()
plt.show()